# Hands-on Tutorial: Building and Evaluating Fair Machine Learning Models for African Healthcare Data

### Author
**Dr. Blessing Ogbuokiri**

Assistant Professor  
Department of Computer Science  
Brock University, St. Catharines, Ontario, Canada

🌐 Director, Responsible AI and Machine Learning Laboratory (RAML Lab)

📧 bogbuokiri@brocku.ca  

---

## Tutorial Description

This hands-on tutorial provides a complete end-to-end workflow for developing, evaluating, and improving fair machine learning models using an African healthcare dataset. Participants will learn how to load and explore healthcare data from Google Drive, build a baseline Logistic Regression model, evaluate predictive performance using standard classification metrics, assess fairness using Demographic Parity Difference (DPD) and Equalized Odds Difference (EOD), visualize fairness metrics, apply Fairlearn bias mitigation techniques, compare model performance before and after mitigation, and implement a simple fairness-aware neural network in PyTorch. The tutorial concludes with a discussion of fairness evaluation when demographic labels are unavailable.

---

### Learning Objectives

By the end of this tutorial, participants will be able to:

- Load and preprocess healthcare datasets in Google Colab.
- Perform exploratory data analysis (EDA).
- Train and evaluate a Logistic Regression classifier.
- Compute Accuracy, Precision, Recall, and F1-score.
- Measure fairness using Demographic Parity Difference (DPD) and Equalized Odds Difference (EOD).
- Apply Fairlearn threshold optimization to reduce demographic disparities.
- Compare predictive performance and fairness before and after mitigation.
- Implement a simple fairness-aware neural network using PyTorch.
- Discuss challenges in evaluating fairness when demographic labels are unavailable.

---

#Install required libraries

In [ ]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q fairlearn

# Import libraries and set random seeds

In [ ]:
# ============================================================
# Import Libraries and Set Random Seeds
# ============================================================

import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    MetricFrame,
    selection_rate,
    true_positive_rate,
    false_positive_rate
)

from fairlearn.postprocessing import ThresholdOptimizer

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("Libraries imported successfully.")

# 1. Loading the Dataset from Google Drive
- Mount Google Drive and load the dataset
- Update the file path if the dataset is stored in another Google Drive folder.

In [ ]:
# ==========================================
# Step 1: Mount Google Drive
# ==========================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# Load Dataset from Google Drive
# ============================================================

from google.colab import drive

drive.mount("/content/drive")
dataset_path = "/content/drive/MyDrive/Health_Equity_Dataset.csv"

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

display(df.head())

# 2. Exploratory Data Analysis

In [ ]:
# ============================================================
# Exploratory Data Analysis
# ============================================================

print("=" * 70)
print("DATASET SHAPE")
print("=" * 70)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\n" + "=" * 70)
print("FIRST FIVE ROWS")
print("=" * 70)
display(df.head())

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)
display(df.dtypes.to_frame(name="Data Type"))

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing_values = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

display(
    missing_values[
        missing_values["Missing Count"] > 0
    ]
)

if df.isnull().sum().sum() == 0:
    print("There are no missing values.")

print("\n" + "=" * 70)
print("DUPLICATED ROWS")
print("=" * 70)
print("Number of duplicated rows:", df.duplicated().sum())

print("\n" + "=" * 70)
print("NUMERICAL SUMMARY")
print("=" * 70)
display(df.describe().T)

print("\n" + "=" * 70)
print("CATEGORICAL SUMMARY")
print("=" * 70)
display(
    df.select_dtypes(
        include=["object", "category"]
    ).describe().T
)

In [ ]:
# ------------------------------------------------------------
# Outcome distribution
# ------------------------------------------------------------
print("\nOutcome distribution:")
display(
    df["Outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)

print("\nOutcome proportions:")
display(
    df["Outcome"]
    .value_counts(normalize=True)
    .rename_axis("Outcome")
    .reset_index(name="Proportion")
)

plt.figure(figsize=(6, 4))

outcome_counts = (
    df["Outcome"]
    .value_counts()
    .sort_index()
)

plt.bar(
    outcome_counts.index.astype(str),
    outcome_counts.values
)

plt.title("Outcome Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Protected attribute distribution
# ------------------------------------------------------------
print("\nSex distribution:")
display(
    df["Sex"]
    .value_counts()
    .rename_axis("Sex")
    .reset_index(name="Count")
)

plt.figure(figsize=(6, 4))

sex_counts = df["Sex"].value_counts()

plt.bar(
    sex_counts.index,
    sex_counts.values
)

plt.title("Distribution of the Protected Attribute")
plt.xlabel("Sex")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Outcome rate by protected group
# ------------------------------------------------------------
outcome_by_sex = (
    df.groupby("Sex")["Outcome"]
    .mean()
    .sort_values()
)

print("\nPositive outcome rate by Sex:")
display(
    outcome_by_sex
    .rename("Positive Outcome Rate")
    .to_frame()
)

plt.figure(figsize=(6, 4))

plt.bar(
    outcome_by_sex.index,
    outcome_by_sex.values
)

plt.title("Positive Outcome Rate by Sex")
plt.xlabel("Sex")
plt.ylabel("Positive Outcome Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Histograms for numerical variables
# ------------------------------------------------------------
numeric_columns = df.select_dtypes(
    include=np.number
).columns.drop(
    ["Outcome"],
    errors="ignore"
)

df[numeric_columns].hist(
    figsize=(16, 12),
    bins=20
)

plt.suptitle(
    "Distribution of Numerical Variables",
    fontsize=16
)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Correlation Matrix with Cell Labels
# ------------------------------------------------------------

correlation_matrix = (
    df.select_dtypes(include=np.number)
    .corr()
)

plt.figure(figsize=(12, 9))

plt.imshow(
    correlation_matrix,
    aspect="auto",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Correlation")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

# Add correlation values inside each cell
for row in range(len(correlation_matrix.index)):
    for column in range(len(correlation_matrix.columns)):

        correlation_value = correlation_matrix.iloc[row, column]

        plt.text(
            column,
            row,
            f"{correlation_value:.2f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

print("EDA completed successfully.")

# 3. Data Preparation and Reproducible Train-Test Split

## a. Define features, target, and protected attribute

- Sex is excluded from the model predictors but retained for fairness analysis.

In [ ]:
# ============================================================
# Define Features, Target and Protected Attribute
# ============================================================

target_column = "Outcome"
protected_attribute = "Sex"

# Exclude the target, protected attribute and identifier
X = df.drop(
    columns=[
        target_column,
        protected_attribute,
        "Patient_ID"
    ]
)

y = df[target_column]

# Retain the protected attribute separately
sensitive_features = df[protected_attribute]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nPredictor variables:")
print(X.columns.tolist())

print("\nProtected attribute:", protected_attribute)

## b. Reproducible train-test split

In [ ]:
# ============================================================
# Reproducible Train-Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Align the protected attribute with retained row indices
sensitive_train = sensitive_features.loc[X_train.index]
sensitive_test = sensitive_features.loc[X_test.index]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining outcome distribution:")
display(
    y_train.value_counts(normalize=True)
    .sort_index()
    .rename("Proportion")
    .to_frame()
)

print("\nTesting outcome distribution:")
display(
    y_test.value_counts(normalize=True)
    .sort_index()
    .rename("Proportion")
    .to_frame()
)

print("\nProtected groups in training data:")
print(sensitive_train.value_counts())

print("\nProtected groups in testing data:")
print(sensitive_test.value_counts())

# 4. Baseline Logistic Regression
-  Build the preprocessing and Logistic Regression pipeline

In [ ]:
# ============================================================
# Baseline Logistic Regression Pipeline
# ============================================================

numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

baseline_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

baseline_model.fit(
    X_train,
    y_train
)

baseline_predictions = baseline_model.predict(
    X_test
)

baseline_probabilities = baseline_model.predict_proba(
    X_test
)[:, 1]

print("Baseline Logistic Regression trained successfully.")

print("\nPredicted class distribution:")
print(
    pd.Series(
        baseline_predictions
    ).value_counts()
)

# 5. Accuracy, Precision, Recall and F1-Score
- Evaluate predictive performance

In [ ]:
# ============================================================
# Baseline Predictive Performance
# ============================================================

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

baseline_precision = precision_score(
    y_test,
    baseline_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_predictions,
    zero_division=0
)

baseline_performance = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    "Score": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1
    ]
})

print("=" * 60)
print("BASELINE LOGISTIC REGRESSION PERFORMANCE")
print("=" * 60)

display(
    baseline_performance.round(4)
)

print("\nClassification Report")
print("=" * 60)

print(
    classification_report(
        y_test,
        baseline_predictions,
        zero_division=0
    )
)

# Confusion matrix
confusion = confusion_matrix(
    y_test,
    baseline_predictions
)

display_confusion = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=baseline_model.classes_
)

display_confusion.plot()

plt.title(
    "Baseline Logistic Regression Confusion Matrix"
)

plt.grid(False)
plt.tight_layout()
plt.show()

#6. Demographic Parity Difference and Equalized Odds Difference
### a. Compute baseline fairness metrics

In [ ]:
# ============================================================
# Baseline Fairness Metrics
# ============================================================

baseline_dpd = demographic_parity_difference(
    y_true=y_test,
    y_pred=baseline_predictions,
    sensitive_features=sensitive_test
)

baseline_eod = equalized_odds_difference(
    y_true=y_test,
    y_pred=baseline_predictions,
    sensitive_features=sensitive_test
)

print("=" * 60)
print("BASELINE FAIRNESS ANALYSIS")
print("=" * 60)

print(
    "Demographic Parity Difference:",
    round(baseline_dpd, 4)
)

print(
    "Equalized Odds Difference:",
    round(baseline_eod, 4)
)

print(
    "\nValues closer to zero indicate "
    "smaller measured disparities."
)

## b. Examine group-specific metrics

In [ ]:
# ============================================================
# Group-Specific Fairness Analysis
# ============================================================

group_metric_frame = MetricFrame(
    metrics={
        "Accuracy": accuracy_score,
        "Selection Rate": selection_rate,
        "True Positive Rate": true_positive_rate,
        "False Positive Rate": false_positive_rate
    },
    y_true=y_test,
    y_pred=baseline_predictions,
    sensitive_features=sensitive_test
)

print("Overall metrics:")
display(
    pd.DataFrame(
        group_metric_frame.overall,
        columns=["Overall"]
    )
)

print("\nMetrics by protected group:")
display(
    group_metric_frame.by_group.round(4)
)

# 7. Fairness Visualizations
##a. Visualize DPD and EOD

In [ ]:
# ============================================================
# Baseline Fairness Visualizations
# ============================================================

baseline_fairness_values = pd.Series({
    "Demographic Parity Difference": baseline_dpd,
    "Equalized Odds Difference": baseline_eod
})

plt.figure(figsize=(8, 5))

bars = plt.bar(
    baseline_fairness_values.index,
    baseline_fairness_values.values
)

plt.title("Baseline Fairness Metrics")
plt.ylabel("Difference")
plt.xlabel("Fairness Metric")
plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1
)

plt.xticks(rotation=10)

for bar, value in zip(
    bars,
    baseline_fairness_values.values
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

## b. Visualize group-specific rates

In [ ]:
# ============================================================
# Group-Specific Rate Visualizations
# ============================================================

group_rates = (
    group_metric_frame.by_group[
        [
            "Selection Rate",
            "True Positive Rate",
            "False Positive Rate"
        ]
    ]
)

group_rates.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Prediction and Error Rates by Sex")
plt.xlabel("Sex")
plt.ylabel("Rate")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

#8. Fairlearn Bias Mitigation

###This section uses Fairlearn’s Threshold Optimizer with an equalized-odds constraint.

## Apply ThresholdOptimizer

In [ ]:
# ============================================================
# Fairlearn Threshold Optimization
# ============================================================

threshold_optimizer = ThresholdOptimizer(
    estimator=baseline_model,
    constraints="equalized_odds",
    objective="accuracy_score",
    predict_method="predict_proba",
    prefit=True
)

# Learn group-specific threshold rules using the training data
threshold_optimizer.fit(
    X_train,
    y_train,
    sensitive_features=sensitive_train
)

# Generate mitigated predictions
mitigated_predictions = threshold_optimizer.predict(
    X_test,
    sensitive_features=sensitive_test,
    random_state=42
)

print("Fairlearn mitigation completed.")

print("\nBaseline prediction distribution:")
print(
    pd.Series(
        baseline_predictions
    ).value_counts()
)

print("\nMitigated prediction distribution:")
print(
    pd.Series(
        mitigated_predictions
    ).value_counts()
)

number_changed = np.sum(
    baseline_predictions
    != mitigated_predictions
)

print(
    "\nNumber of predictions changed:",
    number_changed
)

#9. Before-and-After Fairness Comparison
##a. Compute mitigated performance and fairness metrics

In [ ]:
# ============================================================
# Before-and-After Evaluation
# ============================================================

mitigated_accuracy = accuracy_score(
    y_test,
    mitigated_predictions
)

mitigated_precision = precision_score(
    y_test,
    mitigated_predictions,
    zero_division=0
)

mitigated_recall = recall_score(
    y_test,
    mitigated_predictions,
    zero_division=0
)

mitigated_f1 = f1_score(
    y_test,
    mitigated_predictions,
    zero_division=0
)

mitigated_dpd = demographic_parity_difference(
    y_true=y_test,
    y_pred=mitigated_predictions,
    sensitive_features=sensitive_test
)

mitigated_eod = equalized_odds_difference(
    y_true=y_test,
    y_pred=mitigated_predictions,
    sensitive_features=sensitive_test
)

comparison_results = pd.DataFrame({
    "Before Mitigation": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_dpd,
        baseline_eod
    ],
    "After Mitigation": [
        mitigated_accuracy,
        mitigated_precision,
        mitigated_recall,
        mitigated_f1,
        mitigated_dpd,
        mitigated_eod
    ]
}, index=[
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "DPD",
    "EOD"
])

print("=" * 70)
print("BEFORE-AND-AFTER MITIGATION COMPARISON")
print("=" * 70)

display(
    comparison_results.round(4)
)

print("\nFairness changes:")

print(
    f"DPD: {baseline_dpd:.4f} "
    f"→ {mitigated_dpd:.4f}"
)

print(
    f"EOD: {baseline_eod:.4f} "
    f"→ {mitigated_eod:.4f}"
)

## b. Visualize predictive performance before and after mitigation

In [ ]:
# ============================================================
# Fairness Comparison Visualization
# ============================================================

fairness_comparison = comparison_results.loc[
    [
        "DPD",
        "EOD"
    ]
]

fairness_comparison.plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title(
    "Fairness Before and After Mitigation"
)

plt.xlabel("Fairness Metric")
plt.ylabel("Difference")
plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1
)

plt.xticks(rotation=0)
plt.legend(title="")
plt.tight_layout()
plt.show()

## c. Compare group-specific metrics before and after mitigation

In [ ]:
# ============================================================
# Group-Specific Comparison
# ============================================================

before_group_metrics = MetricFrame(
    metrics={
        "Selection Rate": selection_rate,
        "True Positive Rate": true_positive_rate,
        "False Positive Rate": false_positive_rate
    },
    y_true=y_test,
    y_pred=baseline_predictions,
    sensitive_features=sensitive_test
)

after_group_metrics = MetricFrame(
    metrics={
        "Selection Rate": selection_rate,
        "True Positive Rate": true_positive_rate,
        "False Positive Rate": false_positive_rate
    },
    y_true=y_test,
    y_pred=mitigated_predictions,
    sensitive_features=sensitive_test
)

print("Before mitigation:")
display(
    before_group_metrics.by_group.round(4)
)

print("\nAfter mitigation:")
display(
    after_group_metrics.by_group.round(4)
)

#10. Fairness-Aware PyTorch Model

## The PyTorch model uses:

## Total Loss = Classification Loss + λ × Fairness Penalty

### The fairness penalty measures the difference between the mean predicted positive probabilities for the two protected groups.

## a. Preprocess the data for PyTorch

In [ ]:
# ============================================================
# Prepare Data for PyTorch
# ============================================================

# Fit the preprocessor using training data only
X_train_processed = preprocessor.fit_transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

# Convert sparse matrices to dense arrays when necessary
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()

if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

# Reset indices so arrays align correctly
y_train_array = (
    y_train
    .reset_index(drop=True)
    .to_numpy()
)

y_test_array = (
    y_test
    .reset_index(drop=True)
    .to_numpy()
)

sensitive_train_reset = (
    sensitive_train
    .reset_index(drop=True)
)

sensitive_test_reset = (
    sensitive_test
    .reset_index(drop=True)
)

# Encode Female = 0 and Male = 1
sensitive_mapping = {
    "Female": 0,
    "Male": 1
}

sensitive_train_encoded = (
    sensitive_train_reset
    .map(sensitive_mapping)
    .to_numpy()
)

sensitive_test_encoded = (
    sensitive_test_reset
    .map(sensitive_mapping)
    .to_numpy()
)

# Convert arrays to tensors
X_train_tensor = torch.tensor(
    X_train_processed,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_processed,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_array,
    dtype=torch.float32
).view(-1, 1)

y_test_tensor = torch.tensor(
    y_test_array,
    dtype=torch.float32
).view(-1, 1)

sensitive_train_tensor = torch.tensor(
    sensitive_train_encoded,
    dtype=torch.long
)

print(
    "PyTorch training tensor shape:",
    X_train_tensor.shape
)

print(
    "PyTorch testing tensor shape:",
    X_test_tensor.shape
)

## b. Define the neural network and fairness-aware loss

In [ ]:
# ============================================================
# Define Fairness-Aware Neural Network
# ============================================================

class FairnessAwareNetwork(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)


def demographic_parity_penalty(
    predicted_probabilities,
    sensitive_attribute
):
    """
    Penalizes differences in the average predicted positive
    probability across protected groups.
    """

    group_zero_mask = sensitive_attribute == 0
    group_one_mask = sensitive_attribute == 1

    group_zero_mean = predicted_probabilities[
        group_zero_mask
    ].mean()

    group_one_mean = predicted_probabilities[
        group_one_mask
    ].mean()

    return torch.abs(
        group_zero_mean - group_one_mean
    )

## c. Train standard and fairness-aware PyTorch models

In [ ]:
# ============================================================
# Train PyTorch Models
# ============================================================

def train_pytorch_model(
    fairness_weight=0.0,
    epochs=300,
    learning_rate=0.001
):
    # Reset seed before each model
    torch.manual_seed(42)

    model = FairnessAwareNetwork(
        input_size=X_train_tensor.shape[1]
    )

    classification_loss_function = (
        nn.BCEWithLogitsLoss()
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    training_history = {
        "classification_loss": [],
        "fairness_penalty": [],
        "total_loss": []
    }

    model.train()

    for epoch in range(epochs):

        optimizer.zero_grad()

        logits = model(
            X_train_tensor
        )

        predicted_probabilities = torch.sigmoid(
            logits
        )

        classification_loss = (
            classification_loss_function(
                logits,
                y_train_tensor
            )
        )

        fairness_penalty = (
            demographic_parity_penalty(
                predicted_probabilities,
                sensitive_train_tensor
            )
        )

        total_loss = (
            classification_loss
            + fairness_weight
            * fairness_penalty
        )

        total_loss.backward()
        optimizer.step()

        training_history[
            "classification_loss"
        ].append(
            classification_loss.item()
        )

        training_history[
            "fairness_penalty"
        ].append(
            fairness_penalty.item()
        )

        training_history[
            "total_loss"
        ].append(
            total_loss.item()
        )

    return model, training_history


# Standard neural network
standard_nn, standard_history = train_pytorch_model(
    fairness_weight=0.0
)

# Fairness-aware neural network
fairness_lambda = 2.0

fair_nn, fair_history = train_pytorch_model(
    fairness_weight=fairness_lambda
)

print("PyTorch models trained successfully.")

##d. Generate PyTorch predictions

In [ ]:
# ============================================================
# Generate PyTorch Predictions
# ============================================================

def generate_pytorch_predictions(
    model,
    X_tensor,
    threshold=0.50
):
    model.eval()

    with torch.no_grad():

        logits = model(
            X_tensor
        )

        probabilities = torch.sigmoid(
            logits
        ).numpy().flatten()

    predictions = (
        probabilities >= threshold
    ).astype(int)

    return predictions, probabilities


standard_nn_predictions, standard_nn_probabilities = (
    generate_pytorch_predictions(
        standard_nn,
        X_test_tensor
    )
)

fair_nn_predictions, fair_nn_probabilities = (
    generate_pytorch_predictions(
        fair_nn,
        X_test_tensor
    )
)

print("Standard neural network predictions:")
print(
    pd.Series(
        standard_nn_predictions
    ).value_counts()
)

print("\nFairness-aware neural network predictions:")
print(
    pd.Series(
        fair_nn_predictions
    ).value_counts()
)

## e. Evaluate standard and fairness-aware PyTorch models

In [ ]:
# ============================================================
# Evaluate PyTorch Models
# ============================================================

def evaluate_predictions(
    y_true,
    predictions,
    sensitive_attribute
):
    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),
        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "F1-score": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),
        "DPD": demographic_parity_difference(
            y_true=y_true,
            y_pred=predictions,
            sensitive_features=sensitive_attribute
        ),
        "EOD": equalized_odds_difference(
            y_true=y_true,
            y_pred=predictions,
            sensitive_features=sensitive_attribute
        )
    }


standard_nn_results = evaluate_predictions(
    y_test_array,
    standard_nn_predictions,
    sensitive_test_reset
)

fair_nn_results = evaluate_predictions(
    y_test_array,
    fair_nn_predictions,
    sensitive_test_reset
)

pytorch_comparison = pd.DataFrame({
    "Standard Neural Network":
        standard_nn_results,
    "Fairness-Aware Neural Network":
        fair_nn_results
})

print("=" * 70)
print("PYTORCH MODEL COMPARISON")
print("=" * 70)

display(
    pytorch_comparison.round(4)
)

## f. Visualize PyTorch fairness results

In [ ]:
# ============================================================
# PyTorch Fairness Visualization
# ============================================================

pytorch_comparison.loc[
    [
        "DPD",
        "EOD"
    ]
].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title(
    "Standard and Fairness-Aware "
    "Neural Network Comparison"
)

plt.xlabel("Fairness Metric")
plt.ylabel("Difference")
plt.axhline(
    y=0,
    linestyle="--",
    linewidth=1
)

plt.xticks(rotation=0)
plt.legend(title="")
plt.tight_layout()
plt.show()

## g. Visualize PyTorch training loss

In [ ]:
# ============================================================
# PyTorch Training Loss Visualization
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    standard_history["total_loss"],
    label="Standard Neural Network"
)

plt.plot(
    fair_history["total_loss"],
    label="Fairness-Aware Neural Network"
)

plt.title("PyTorch Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.legend()
plt.tight_layout()
plt.show()

## Fairness Evaluation When Demographic Labels Are Unavailable

Fairness metrics such as Demographic Parity Difference and Equalized Odds Difference require demographic or sensitive-group information. When protected attributes such as sex, gender, race, ethnicity, disability status, or age group are unavailable, it is not possible to directly determine whether model outcomes or error rates are equitable across those groups.

A model should not be described as fair merely because demographic disparities could not be measured. The appropriate conclusion is that fairness across protected demographic groups could not be directly evaluated using the available data.

One possible approach is to collect demographic information through a voluntary, transparent, privacy-preserving, and ethically approved process. Researchers should clearly explain why the information is being collected, how it will be protected, and how it will be used to evaluate and improve the system.

When direct demographic labels cannot be collected, researchers may evaluate model performance across available and contextually relevant subgroups. Examples include geographic region, language variety, healthcare facility, device type, socioeconomic indicator, or data source. However, these variables should not automatically be treated as substitutes for protected characteristics because proxy variables may be inaccurate and may reinforce stereotypes.

Error-slicing methods can also be used to identify subsets of the data where the model performs poorly. These methods examine combinations of features and search for groups with unusually high error rates. Representation analysis, clustering, worst-group evaluation, and influence analysis may also reveal hidden areas of model weakness.

Counterfactual testing provides another indirect method. Researchers can create matched examples that differ only in identity terms, names, dialect features, or other socially meaningful characteristics. A substantial change in prediction between otherwise similar examples may indicate individual or behavioural bias.

These indirect approaches are useful for identifying potential problems, but they do not replace fairness evaluation using real demographic groups. Therefore, fairness assessment without demographic labels should combine multiple auditing methods, stakeholder consultation, careful documentation, privacy-preserving data practices, and plans for improved data collection.

In [ ]:
# ============================================================
# Final Tutorial Summary
# ============================================================

print("=" * 70)
print("FINAL TUTORIAL SUMMARY")
print("=" * 70)

print("\nBaseline Logistic Regression")
print(
    comparison_results[
        ["Before Mitigation"]
    ].round(4)
)

print("\nFairlearn-Mitigated Logistic Regression")
print(
    comparison_results[
        ["After Mitigation"]
    ].round(4)
)

print("\nPyTorch Comparison")
print(
    pytorch_comparison.round(4)
)

print("\nKey interpretation:")
print(
    "- DPD and EOD values closer to zero "
    "indicate smaller measured disparities."
)

print(
    "- Fairness mitigation may improve "
    "fairness while changing predictive performance."
)

print(
    "- The fairness-aware PyTorch loss illustrates "
    "how fairness can be incorporated during training."
)

print(
    "- Different fairness definitions may produce "
    "different trade-offs."
)